In [1]:
from pathlib import Path
from dotenv import load_dotenv
from chunk_pipeline import load_markdown_pages, merge_markdown_pages

load_dotenv()

pages = load_markdown_pages("output")
if not pages:
    raise FileNotFoundError("未找到 output/doc_*.md，请先执行解析步骤")

raw_text = merge_markdown_pages(pages)
print(f"已加载 {len(pages)} 个 markdown 页面，总长度: {len(raw_text)} 字符")


In [2]:
from openai import OpenAI
import re
import json
import os

# ── 配置区：全部从环境变量读取，本地开发在 .env 里设置即可 ──────────────────────
#
#   必填：
#     LLM_API_KEY     你的 API Key
#
#   可选（有默认值）：
#     LLM_BASE_URL    不填则走 OpenAI 官方地址；填 DeepSeek 就写 https://api.deepseek.com
#     LLM_MODEL       默认 gpt-4o，DeepSeek 就填 deepseek-chat

client = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.environ.get("LLM_BASE_URL") or None,
)
MODEL = os.environ.get("LLM_MODEL", "gpt-4o")

In [3]:
# ── Step 0：提取标题树，预览文档结构 ─────────────────────────────────────────

def extract_headings(text: str) -> list[dict]:
    """
    扫描全文，提取每个标题的层级、文本、行号。
    返回格式：[{"level": 2, "text": "实习经历", "line": 12}, ...]
    """
    headings = []
    for line_no, line in enumerate(text.splitlines()):
        match = re.match(r"^(#{1,6})\s+(.+)", line)
        if match:
            headings.append({
                "level": len(match.group(1)),
                "text":  match.group(2).strip(),
                "line":  line_no,
            })
    return headings


headings = extract_headings(raw_text)

print(f"一共找到 {len(headings)} 个标题：\n")
for h in headings:
    print("  " * (h["level"] - 1) + f"- H{h['level']}: {h['text']}")

一共找到 5 个标题：

- H1: 范逸
  - H2: 教育背景
  - H2: 实习经历
  - H2: 美的集团·中央研究院
  - H2: 项目经历


In [4]:
# ── Step 1：LLM 审查标题树，修正层级错误 ──────────────────────────────────────
#
# 不做规则预筛，直接交给 LLM 凭语义判断，对任意文档类型都成立。

def llm_verify_headings(all_headings: list[dict]) -> list[dict]:
    """
    把完整标题树喂给 LLM，返回需要修正的标题列表。
    格式：[{"text": "...", "correct_level": 3}, ...]
    """
    if not all_headings:
        return []

    tree_str = "\n".join(
        f"{'  ' * (h['level'] - 1)}H{h['level']}: {h['text']}"
        for h in all_headings
    )

    prompt = f"""下面是一份从 PDF 解析出来的 Markdown 标题树。
PDF 解析器有时会把标题层级识别错——常见错误是某个标题在语义上应该是上一个标题的子项，却被识别成了同级。

标题树：
{tree_str}

请仔细阅读这棵树，找出所有语义层级与当前 H 级别不符的标题。
判断标准：只看语义——如果一个标题描述的是某个上级标题下的具体实例或细节，它就应该是子级。

只返回需要修正的标题，格式为 JSON 数组，不要输出任何其他内容：
[
  {{"text": "标题原文", "correct_level": 正确的层级数字}},
  ...
]

如果整棵树层级都正确，返回空数组 []。"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    raw = resp.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\[.*\]", raw, re.DOTALL)
        return json.loads(match.group()) if match else []


corrections = llm_verify_headings(headings)
correction_map = {c["text"]: c["correct_level"] for c in corrections}

# 打印纠正明细
if corrections:
    print(f"\n{'='*60}")
    print(f"LLM 共发现 {len(corrections)} 处标题层级错误：")
    for c in corrections:
        original = next((h for h in headings if h["text"] == c["text"]), None)
        original_level = original["level"] if original else "?"
        print(f"  H{original_level} → H{c['correct_level']}  |  {c['text']}")
    print(f"{'='*60}\n")
else:
    print("LLM 未发现标题层级错误，无需修正。")


LLM 共发现 1 处标题层级错误：
  H2 → H3  |  美的集团·中央研究院



In [5]:
# ── Step 1b：把修正应用到原文 ─────────────────────────────────────────────────

def apply_corrections(text: str, correction_map: dict) -> str:
    """
    按行扫描原文，遇到需要修正的标题就替换成正确层级的 '#'，其余行原样保留。
    """
    fixed_lines = []
    for line in text.splitlines():
        match = re.match(r"^(#{1,6})\s+(.+)", line)
        if match:
            heading_text = match.group(2).strip()
            if heading_text in correction_map:
                new_level = correction_map[heading_text]
                line = "#" * new_level + " " + heading_text
        fixed_lines.append(line)
    return "\n".join(fixed_lines)


fixed_text = apply_corrections(raw_text, correction_map)

In [6]:
# ── Step 2：切割 chunk，同时识别哪些 chunk 含表格 ──────────────────────────────
#
# 切割逻辑本身不变，只是在生成 chunk 时顺手标记 has_table。
# 含表格的 chunk 后续走 JSON 提取，不进 embedding。

def split_into_chunks(text: str) -> list[dict]:
    """
    按标题切割文本，每个标题就是一个切割点。
    每个 chunk 额外带一个 has_table 字段，标记内容里是否含有 HTML 表格。
    """
    lines = text.splitlines()
    chunks = []
    current_chunk_lines = []
    current_heading = None
    heading_stack: list[dict] = []

    def flush(stack):
        """把当前积累的行打包成一个 chunk，顺手检测是否含表格。"""
        content = "\n".join(current_chunk_lines).strip()
        if not content:
            return None
        return {
            "heading_path": " > ".join(h["text"] for h in stack),
            "content":      content,
            # <table 出现就认定含表格，简单可靠
            "has_table":    "<table" in content,
        }

    for line in lines:
        match = re.match(r"^(#{1,6})\s+(.+)", line)

        if not match:
            current_chunk_lines.append(line)
            continue

        level       = len(match.group(1))
        heading_text = match.group(2).strip()

        if current_heading is not None:
            chunk = flush(heading_stack)
            if chunk:
                chunks.append(chunk)
            current_chunk_lines = []

        while heading_stack and heading_stack[-1]["level"] >= level:
            heading_stack.pop()
        heading_stack.append({"level": level, "text": heading_text})

        current_heading = {"level": level, "text": heading_text}
        current_chunk_lines.append(line)

    # 最后一个 chunk
    chunk = flush(heading_stack)
    if chunk:
        chunks.append(chunk)

    return chunks


chunks = split_into_chunks(fixed_text)

text_chunks  = [c for c in chunks if not c["has_table"]]
table_chunks = [c for c in chunks if c["has_table"]]

print(f"共 {len(chunks)} 个 chunk：{len(text_chunks)} 个纯文本，{len(table_chunks)} 个含表格")

共 5 个 chunk：5 个纯文本，0 个含表格


In [7]:
# ── Step 3：对含表格的 chunk，用 LLM 提取结构化 JSON ──────────────────────────
#
# 表格不进 embedding，单独存成 JSON，查询时走精确匹配。
# LLM 负责把 HTML 表格转成字段清晰的 JSON 数组，
# 字段名由 LLM 根据表格内容自行推断，不做硬编码。

def extract_table_as_json(chunk: dict) -> list[dict]:
    """
    把含表格的 chunk 喂给 LLM，提取成 JSON 数组。
    每一行表格对应一个 JSON 对象，字段名从表头推断。
    返回的每条记录额外带上来源信息，方便溯源。
    """
    prompt = f"""下面是一段包含 HTML 表格的文本，来自文档章节：{chunk['heading_path']}

{chunk['content']}

请把表格内容提取成 JSON 数组，要求：
1. 每行表格数据对应一个 JSON 对象
2. 字段名用英文，根据表头语义自行命名，简洁清晰
3. 字段值保留原文，不要改写
4. 只返回 JSON 数组，不要输出任何其他内容

"""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    raw = resp.choices[0].message.content.strip()
    try:
        rows = json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r"\[.*\]", raw, re.DOTALL)
        rows = json.loads(m.group()) if m else []

    # 每条记录带上来源，查询结果可以溯源到具体章节
    for row in rows:
        row["_source_heading"] = chunk["heading_path"]

    return rows


# 对所有含表格的 chunk 逐一提取
all_table_records = []
for chunk in table_chunks:
    records = extract_table_as_json(chunk)
    all_table_records.extend(records)
    print(f"\n章节: {chunk['heading_path']}")
    print(f"提取到 {len(records)} 条记录，示例：")
    print(json.dumps(records[0], ensure_ascii=False, indent=2) if records else "  (空)")

In [8]:
# ── Step 4：预览结果 ──────────────────────────────────────────────────────────

print("\n── 纯文本 chunks（将进入 embedding）──")
for i, chunk in enumerate(text_chunks):
    print(f"\n{'='*60}")
    print(f"Chunk {i+1}  |  路径: {chunk['heading_path']}")
    print(f"{'='*60}")
    print(chunk["content"][:300], "..." if len(chunk["content"]) > 300 else "")

print("\n\n── 表格 JSON（将走精确匹配）──")
print(json.dumps(all_table_records, ensure_ascii=False, indent=2))


── 纯文本 chunks（将进入 embedding）──

Chunk 1  |  路径: 范逸
# 范逸

15601792659 | 15601792659@163.com | 男 | 中共党员

<div style="text-align: center;"><img src="imgs/img_in_image_box_973_21_1093_163.jpg" alt="Image" width="10%" /></div> 

Chunk 2  |  路径: 范逸 > 教育背景
## 教育背景

新南威尔士大学·信息技术（数据科学与工程）

碩士

主修课程：深度学习与神经网络、机器学习、统计推断、图算法与图神经网络、大数据技术

河南理工大学·计算机科学与技术

2023.09 – 2025.10

学士

2019.09 – 2023.07

两次获得国家励志奖学金、一次孙越崎一等奖学金、两次三好学生荣誉称号，英语 CET-6 542 分，IELTS 6.5 分 

Chunk 3  |  路径: 范逸 > 实习经历
## 实习经历

北京抖音信息服务有限公司

推荐算法

2025.05 - 2025.07

➢ 精排任务需同时优化完播率与观看时长，两目标在训练中存在冲突，且时长偏差使长视频更难完播，分发结果对短视频偏置明显

在兼顾完播率与观看时长的前提下缓解多任务冲突，降低强统计特征过拟合，提升长视频与弱历史样本的泛化能力。

从共享底层 DNN 加双塔头基线出发，引入 MMoE 进行任务表示分离；通过特征重要性与分桶实验定位全局热度等稠密特征过强导致 ID 与内容特征贡献受抑；在 MMoE 上叠加 SENet 重加权与双线性交叉，增强热度内容上下文 ID 特征的有效交互。

离线结果达到完播率 AU ...

Chunk 4  |  路径: 范逸 > 实习经历 > 美的集团·中央研究院
### 美的集团·中央研究院

研发工程师

2024.12 - 2025.02

空调实验测试端在初期存在实验参数主要依赖手工录入，实时监控与历史查询链路分散，页面长时间运行后易出现图表错位和性能下降三类问题；同时工况智能分析能力尚未落地。

在既有系统上完成“参数配置-实验执行-实时监控-历史导出”落地，并推进工况智能分析对话模块的前端原型验证。



In [9]:
# ── Step 5：Embedding + 存入 Qdrant（重建一次，启用 BM25/BM42）──────────────
#
# 说明：你之前的 documents 可能是 dense-only schema。
# 这里先删再建，确保 sparse 路由（bm25/bm42）生效。

import os
from pathlib import Path
from embedding import get_embedder, upsert_chunks, _make_qdrant_client

# 可按需开关
os.environ["ENABLE_BM25"] = "1"
os.environ["ENABLE_BM42"] = "1"

collection_name = "documents"
qdrant = _make_qdrant_client()

# 第一次接入 BM25/BM42 时建议重建
try:
    qdrant.delete_collection(collection_name)
    print(f"已删除旧 collection: {collection_name}")
except Exception:
    pass

# doc_id 用文件名，多文档入库时方便过滤
doc_id = Path("output/doc_0.md").stem

# 语言选项："en" / "zh" / "mixed" / "auto"
embedder = get_embedder(doc_language="mixed")

upsert_chunks(
    chunks=text_chunks,
    embedder=embedder,
    collection_name=collection_name,
    doc_id=doc_id,
)

# 看看当前 collection 路由是否就绪
from retrieval import _collection_route_info
print(_collection_route_info(qdrant, collection_name))


已删除旧 collection: documents
警告: FlagEmbedding 初始化失败，将自动回退到 sentence-transformers。 原因: ImportError: cannot import name 'is_torch_fx_available' from 'transformers.utils.import_utils' (/Users/felix/Desktop/powerful-rag/.venv/lib/python3.12/site-packages/transformers/utils/import_utils.py)


Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-zh-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


已创建 collection: documents，向量维度: 512，sparse_routes=['bm25', 'bm42']
正在 embedding 5 个 chunks...
已写入 5 条向量到 collection: documents，sparse_routes=['bm25', 'bm42']
{'dense_using': None, 'bm25': True, 'bm42': True}


In [10]:
# ── Step 5.1：查看已入库的 payload 与向量（不硬编码 point id）──────────────────

from qdrant_client import QdrantClient

q = QdrantClient(host="localhost", port=6333, proxy=None, trust_env=False, check_compatibility=False)

points, _ = q.scroll(
    collection_name="documents",
    limit=2,
    with_vectors=True,
    with_payload=True,
)

print("points:", len(points))
for p in points:
    print("\npoint id:", p.id)
    print("payload keys:", list((p.payload or {}).keys()))

    vec = p.vector
    if isinstance(vec, dict):
        dense = vec.get("") or vec.get("default")
        print("vector keys:", list(vec.keys()))
        if dense is not None:
            print("dense dim:", len(dense), "first10:", dense[:10])

        bm25 = vec.get("bm25")
        bm42 = vec.get("bm42")
        if bm25 is not None:
            print("bm25 type:", type(bm25).__name__)
        if bm42 is not None:
            print("bm42 type:", type(bm42).__name__)
    else:
        print("dense dim:", len(vec), "first10:", vec[:10])


points: 2

point id: 00329a8a-8852-4691-b316-a8dedbfd0036
payload keys: ['doc_id', 'chunk_index', 'heading_path', 'content']
vector keys: ['bm25', '', 'bm42']
dense dim: 512 first10: [-0.019678907, -0.049603876, 0.03459668, -0.05025316, -0.035231583, 0.019103296, -0.06151338, 0.020379346, -0.003750411, 0.012173067]
bm25 type: SparseVector
bm42 type: SparseVector

point id: 56716a3f-7b64-4fac-b622-5c299c185dc5
payload keys: ['doc_id', 'chunk_index', 'heading_path', 'content']
vector keys: ['bm42', '', 'bm25']
dense dim: 512 first10: [-0.010523187, 0.041906375, -0.0029893161, -0.024075806, -0.038172346, -0.024350308, -0.03385061, 0.026891228, 0.02219548, -0.007920944]
bm25 type: SparseVector
bm42 type: SparseVector


In [11]:
import importlib, retrieval
importlib.reload(retrieval)


<module 'retrieval' from '/Users/felix/Desktop/powerful-rag/notebooks/retrieval.py'>

In [12]:
# ── Step 6：Hybrid 检索测试 ───────────────────────────────────────────────────
#
# 主链路：dense + bm25 + bm42（collection 支持时） -> RRF 融合
#       + table route（可选） -> RRF 融合 -> Cross-Encoder rerank

from retrieval import hybrid_search, print_results

query = "范逸有哪些项目经历"

results = hybrid_search(
    query=query,
    embedder=embedder,               # Step 5 初始化的 embedder
    collection_name="documents",
    table_records=all_table_records, # Step 3 提取的表格数据；没有就传 []
    top_k=5,
)

print_results(results)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


#1  来源: vector_hybrid  |  Final: 0.998346  |  Rerank: 0.998346  |  RRF: 0.0161  |  路径: 范逸 > 项目经历
RRF构成(总融合): vector@rank2:0.016129
RRF构成(向量内部): dense@rank5:0.015385 + bm42@rank2:0.016129
Reranker模型: BAAI/bge-reranker-v2-m3
## 项目经历

Orchestrix AI（n8n-like 可视化 AI 工作流平台）

2025.09–2025.11

• 项目背景：工作流平台的核心价值是将跨系统、多步骤、异步流程标准化，降低人工操作成本，提高流程的稳定性、可追溯性和复用效率。计划通过完整自研验证其底层机制，提高自己对 AI 应用工程化落地的理解深度。

- 核心任务：以独立项目方式实现一个 AI 赋能的、类似于 n8n 的工作流平台，覆盖编排、执行、观测、调度和外部能力集成，形成可运行的端到端闭环，沉淀可复用的工程方法。

· 项目内容:

1. 从 0 到 1 完成了平台核心架构与实现，基于 API Route  ...

#2  来源: vector_hybrid  |  Final: 0.966183  |  Rerank: 0.966183  |  RRF: 0.0164  |  路径: 范逸 > 实习经历
RRF构成(总融合): vector@rank1:0.016393
RRF构成(向量内部): dense@rank3:0.015873 + bm42@rank1:0.016393
Reranker模型: BAAI/bge-reranker-v2-m3
## 实习经历

北京抖音信息服务有限公司

推荐算法

2025.05 - 2025.07

➢ 精排任务需同时优化完播率与观看时长，两目标在训练中存在冲突，且时长偏差使长视频更难完播，分发结果对短视频偏置明显

在兼顾完播率与观看时长的前提下缓解多任务冲突，降低强统计特征过拟合，提升长视频与弱历史样本的泛化能力。

从共享底层 DNN 加双塔头基线出发，引入 MMoE 进行任务表示分离；通过特征重要性与分桶实验定位全局热度等稠密特征过强导致 ID 与内容特征贡献受抑；在 MMoE 上叠

In [13]:
# ── Step 7：Generation ────────────────────────────────────────────────────────
#
# 把检索结果喂给 LLM，生成带引用溯源的最终答案。
# stream=True 边生成边打印，stream=False 等完整结果。

from generation import generate, print_response

question = "范逸有哪些项目经历"

# results 直接复用上一步 hybrid_search 的结果
response = generate(
    question=question,
    results=results,
    stream=True,
)

print_response(response)


── 生成答案 ──────────────────────────────────────────
范逸的项目经历包括：

1. **Orchestrix AI（n8n-like 可视化 AI 工作流平台）**：在2025年9月至2025年11月期间，范逸独立实现了一个AI赋能的工作流平台，涵盖编排、执行、观测、调度和外部能力集成等功能，最终交付了一个可执行的端到端工作流平台【参考资料1】。

2. **AI 智能体开发（LangChain/LangGraph+MCP 多工具编排）**：在2025年11月至2026年1月期间，范逸开发了一个命令行Agent原型，实现了从模型调用到外部工具执行的端到端集成验证，完成了可运行的Agent原型与工具接入模板【参考资料1】。


问题: 范逸有哪些项目经历

范逸的项目经历包括：

1. **Orchestrix AI（n8n-like 可视化 AI 工作流平台）**：在2025年9月至2025年11月期间，范逸独立实现了一个AI赋能的工作流平台，涵盖编排、执行、观测、调度和外部能力集成等功能，最终交付了一个可执行的端到端工作流平台【参考资料1】。

2. **AI 智能体开发（LangChain/LangGraph+MCP 多工具编排）**：在2025年11月至2026年1月期间，范逸开发了一个命令行Agent原型，实现了从模型调用到外部工具执行的端到端集成验证，完成了可运行的Agent原型与工具接入模板【参考资料1】。

── 引用来源 ──────────────────────────────────────────
  [1] 范逸 > 项目经历  (相关度: 0.998)
  [2] 范逸 > 实习经历  (相关度: 0.966)
  [3] 范逸 > 实习经历 > 美的集团·中央研究院  (相关度: 0.943)
  [4] 范逸 > 教育背景  (相关度: 0.433)
  [5] 范逸  (相关度: 0.056)
